# Klasifikasi Tangisan Bayi (Hanya Model SVM)

Notebook ini merupakan versi yang telah disederhanakan dan diterjemahkan ke dalam Bahasa Indonesia dari versi sebelumnya. Notebook ini secara khusus **hanya menggunakan model Support Vector Machine (SVM)**.

Terdapat perbaikan penting di sini: Kita menggunakan **ekstraksi fitur MFCC (13 koefisien)** dan mengubahnya menjadi 1D array (`np.mean`), sehingga proses pelatihan jauh lebih cepat, hemat memori, dan yang paling penting **100% kompatibel dengan file `main.py` di backend API Anda**.

In [ ]:
import os
import glob
import numpy as np
import librosa
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import joblib
from tqdm.notebook import tqdm

### 1. Konfigurasi Dataset

In [ ]:
# Menentukan path direktori dataset
direktori = "donateacry-corpus/donateacry_corpus_cleaned_and_updated_data/"
model_save_path = "svm_model_bayi_mfcc.pkl"

if os.path.exists(direktori):
    alasan_menangis = next(os.walk(direktori))[1]
else:
    print(f"Direktori {direktori} tidak ditemukan. Menggunakan label default.")
    alasan_menangis = ['belly_pain', 'burping', 'discomfort', 'hungry', 'tired']

print(f"Kategori yang ditemukan: {alasan_menangis}")

### 2. Fungsi Ekstraksi MFCC
Fungsi ini sengaja dibuat persis seperti yang ada di `backend/main.py` agar fitur yang digunakan untuk melatih dan memprediksi benar-benar sama.

In [ ]:
def extract_mfcc(file_path, sr=16000, n_mfcc=13):
    try:
        # Memuat audio
        y, sr_loaded = librosa.load(file_path, sr=sr)
        # Ekstraksi MFCC
        mfccs = librosa.feature.mfcc(y=y, sr=sr_loaded, n_mfcc=n_mfcc)
        # Mengambil rata-rata agar menjadi array 1D (hanya 13 fitur)
        mfccs_mean = np.mean(mfccs, axis=1) 
        return mfccs_mean
    except Exception as e:
        print(f"Gagal memproses {file_path}: {e}")
        return None

### 3. Memuat Data dan Ekstraksi Fitur

In [ ]:
X = []
y_teks = []

print("Memulai proses ekstraksi fitur MFCC...")
for label in alasan_menangis:
    print(f"-> Memuat dan memproses data untuk kategori: {label}")
    file_audio = glob.glob(os.path.join(direktori, label, "*"))
    
    for file in tqdm(file_audio, desc=label):
        fitur = extract_mfcc(file)
        if fitur is not None:
            X.append(fitur)
            y_teks.append(label)

X = np.array(X)
y_teks = np.array(y_teks)

print(f"\nTotal data audio yang berhasil diproses: {len(X)} sampel")

### 4. Konversi Label Teks Menjadi Numerik dan Pembagian Data

In [ ]:
kategori_unik = np.unique(y_teks)
y = np.zeros(len(y_teks))

for i, label in enumerate(y_teks):
    y[i] = np.where(kategori_unik == label)[0][0]

# Membagi data menjadi data latih (75%) dan data uji (25%)
X_latih, X_uji, y_latih, y_uji = train_test_split(X, y, test_size=0.25, random_state=42)
print(f"Jumlah Data Latih: {len(X_latih)}")
print(f"Jumlah Data Uji: {len(X_uji)}")

### 5. Melatih Model Support Vector Machine (SVM)

In [ ]:
print("Mulai melatih model SVM...")

# Gunakan probability=True agar predict_proba (confidence score) bisa dipakai di backend
model_svm = SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42)
model_svm.fit(X_latih, y_latih)

print("Model selesai dilatih!")

### 6. Evaluasi Akurasi Model

In [ ]:
prediksi = model_svm.predict(X_uji)
akurasi = accuracy_score(y_uji, prediksi)

print(f"Akurasi Model SVM: {akurasi * 100:.2f}%\n")
print("Laporan Klasifikasi:")
print(classification_report(y_uji, prediksi, target_names=kategori_unik))

### 7. Menyimpan Model

In [ ]:
joblib.dump(model_svm, model_save_path)
print(f"Selesai! Model siap digunakan dan telah disimpan sebagai: '{model_save_path}'")